In [84]:
import random
import torch
import io
import pyarrow as pa
import os
import copy
import pytorch_lightning as pl
from sacred import Experiment
from PIL import Image
from tqdm import tqdm
import numpy as np
import skimage.io as skio
import matplotlib.pyplot as plt
from refer import REFER

from torch.optim import AdamW

from transformers import ElectraTokenizer

from refcoco_utils import get_bounded_subimage
from refcoco_utils import _config
from refcoco_utils import _loss_names

from meter.transforms import keys_to_transforms
from meter.config import ex
from meter.modules import METERTransformerSS
from meter.datamodules.multitask_datamodule import MTDataModule
from meter.datasets.base_dataset import BaseDataset

### Data and Model

In [2]:
data_root = '/home/claytonfields/nlp/code/data/coco'  # contains refclef, refcoco, refcoco+, refcocog and images
dataset = 'refcoco' 
splitBy = 'unc'
refer = REFER(data_root, dataset, splitBy)

loading dataset refcoco into memory...
testing
creating index...
index created.
DONE (t=10.45s)


In [3]:
_config = copy.deepcopy(_config)
pl.seed_everything(_config["seed"])

dm = MTDataModule(_config, dist=False)
model = METERTransformerSS(_config)

Global seed set to 0
Some weights of the model checkpoint at google/electra-small-discriminator were not used when initializing ElectraModel: ['discriminator_predictions.dense.bias', 'discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense.weight', 'discriminator_predictions.dense_prediction.weight']
- This IS expected if you are initializing ElectraModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ElectraModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


### Data Class

In [60]:
class RefcocoDataset(torch.utils.data.Dataset):

    def __init__(self, refer, tokenizer, split='', max_bb = 75):
        self.tokenizer = tokenizer
        self.refer = refer
        self.max_bb = max_bb
        self.split = split
        self.sent_ids = self.get_sent_ids()
        self.duds = []
        

    def __len__(self):
        return len(self.sent_ids)
    
    def get_sent_ids(self):
        sent_ids = []
        for ref_id in self.refer.getRefIds(split=self.split):
            ref = self.refer.Refs[ref_id]
            for sent_id in ref['sent_ids']:
                sent_ids.append(sent_id)
        return sent_ids

    def __getitem__(self, index):
        sent_id = self.sent_ids[index]
        ref = self.refer.sentToRef[sent_id]
        sent = self.refer.Sents[sent_id]
        
        img_id = ref['image_id']
        ann_id = ref['ann_id']
        objs = self.refer.imgToAnns[img_id]
        obj_ids = [obj['id'] for obj in objs]
        
        sub_images = []
        for obj in objs:
            try:
                x_a = get_bounded_subimage(refer, img_id, obj['id'], xs=224,ys=224, show=False)
            except ValueError:
                print(f'ValueError at setence id: {sent_id}')
                self.duds.append(sent_id)
                break
                
            if x_a is not None:
                sub_images.append(x_a)
        num_sub_images = len(sub_images)      
            
        text_ids = tokenizer.encode(
            sent['sent'],
            padding="max_length",
            truncation=True,
            max_length=40,
            return_special_tokens_mask=True,
        )
        text_masks = [1 if text_ids[i]>0 else 0 for i,_ in enumerate(text_ids)]
        text_labels = [[-100 for i in range(40)]]

        ids = [text_ids for i in range(num_sub_images)]
        masks = [text_masks for _ in range(num_sub_images)]
        labels = [text_labels for i in range(num_sub_images)]
            
        return_dict = {
            'ann_id' : ann_id,
            'image' : torch.cat(sub_images),
            'num_bb' : num_sub_images,
            'obj_ids' : obj_ids,
            'sent_id' : sent_id,
            'text' : sent['sent'],
            'text_ids' : torch.tensor(ids),
            'text_labels' : torch.tensor(labels),
            'text_masks' : torch.tensor(masks)
        }  

        return return_dict
    
    def collate_fn(self, batch):
        num_bb = [example['num_bb'] for example in batch]
        max_bb = max(num_bb)
        ann_id =  [example['ann_id'] for example in batch]
        image = [example['image'] for example in batch]
        sent_id = [example['sent_id'] for example in batch]
        text = [example['text'] for example in batch]
        text_ids = [example['text_ids'] for example in batch]
        text_labels = [example['text_labels'] for example in batch]
        text_masks = [example['text_masks'] for example in batch]
        
        return_dict = {
            'ann_id' : ann_id,
            'image' : images,
            'num_bb' : num_bb,
            'obj_ids' : obj_ids,
            'sent_id' : sent_id,
            'text' : text,
            'text_ids' : text_ids,
            'text_labels' : text_labels,
            'text_masks' : text_masks
        }  
        
        

The collate function below seems not to work, buts all the values in lists. May be useful for iterating through each example. Another approach is probably better tho.

In [72]:
def collate_fn(batch):
        num_bb = [example['num_bb'] for example in batch]
        max_bb = max(num_bb)
        ann_id =  [example['ann_id'] for example in batch]
        image = [example['image'] for example in batch]
        obj_ids = [example['obj_ids'] for example in batch]
        sent_id = [example['sent_id'] for example in batch]
        text = [example['text'] for example in batch]
        text_ids = [example['text_ids'] for example in batch]
        text_labels = [example['text_labels'] for example in batch]
        text_masks = [example['text_masks'] for example in batch]
        
        return_dict = {
            'ann_id' : ann_id,
            'image' : images,
            'num_bb' : num_bb,
            'obj_ids' : obj_ids,
            'sent_id' : sent_id,
            'text' : text,
            'text_ids' : text_ids,
            'text_labels' : text_labels,
            'text_masks' : text_masks
        }  
        return return_dict

This function simply returns a list of examples in a dictionary. This will work for now as a simple solution. More will probably need to be done later. 

In [90]:
def collate_fn(batch):
    return batch

In [91]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
optimizer = AdamW(model.parameters(), lr=1e-4)
# Ref Res with METER
tokenizer = ElectraTokenizer.from_pretrained('google/electra-small-discriminator')
BATCH_SIZE = 10

epochs = 1

ds = RefcocoDataset(refer, tokenizer)

In [92]:
ds = RefcocoDataset(refer, tokenizer)
train_params = {'batch_size': BATCH_SIZE,
                'shuffle': False,
                'num_workers': 0,
                'collate_fn' : collate_fn
                }

training_loader = torch.utils.data.DataLoader(ds, **train_params)


In [93]:
for i, data in enumerate(training_loader):
    if i ==1:
        break
data

[{'ann_id': 485695,
  'image': tensor([[[[0.7647, 0.7451, 0.7098,  ..., 0.9529, 0.9490, 0.9412],
            [0.7490, 0.7412, 0.7255,  ..., 0.9765, 0.9569, 0.9412],
            [0.7137, 0.7216, 0.7333,  ..., 0.9765, 0.9529, 0.9412],
            ...,
            [0.8157, 0.7961, 0.7686,  ..., 0.8353, 0.8118, 0.8000],
            [0.7804, 0.7725, 0.7686,  ..., 0.7686, 0.7804, 0.7922],
            [0.7098, 0.7294, 0.7647,  ..., 0.7843, 0.8000, 0.8157]],
  
           [[0.6941, 0.6706, 0.6235,  ..., 0.8941, 0.8863, 0.8784],
            [0.6745, 0.6667, 0.6392,  ..., 0.9216, 0.8980, 0.8824],
            [0.6392, 0.6471, 0.6510,  ..., 0.9255, 0.9020, 0.8863],
            ...,
            [0.7529, 0.7333, 0.7059,  ..., 0.7647, 0.7373, 0.7294],
            [0.7176, 0.7098, 0.7059,  ..., 0.6980, 0.7020, 0.7176],
            [0.6471, 0.6667, 0.7020,  ..., 0.7216, 0.7333, 0.7451]],
  
           [[0.4588, 0.4392, 0.4078,  ..., 0.8275, 0.7922, 0.7725],
            [0.4431, 0.4353, 0.4196,  ..., 0.

In [76]:
model

METERTransformerSS(
  (cross_modal_text_transform): Linear(in_features=256, out_features=256, bias=True)
  (cross_modal_image_transform): Linear(in_features=192, out_features=256, bias=True)
  (token_type_embeddings): Embedding(2, 256)
  (vit_model): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (blocks): ModuleList(
      (0): Block(
        (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=192, out_features=576, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=192, out_features=192, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_featu

#### Training Loop

In [94]:
# model.to(device)
model.train()

for batch in tqdm(training_loader):
    optimizer.zero_grad()
    targets = []
    logit_list = []
    for b in batch:
        sent_id = b['sent_id']
        infer_dict = model.infer(b)
        logits = model.ref_classifier(infer_dict['cls_feats'])
        logit_list.append(logits)
        obj_ids = b['obj_ids']
        ann_id = b['ann_id']

        target = [obj_ids.index(ann_id)]#.to(device)
        targets.append(target)
    logit_tensor = torch.tensor(logit_list)
    target_tensor = torch.tensor(targets)
    loss = loss_fn(logits.reshape(1,-1),target)
    losses.append(loss.item())
    loss.backward()

    optimizer.step()
    


  0%|                                                 | 0/14221 [00:03<?, ?it/s]


ValueError: not enough values to unpack (expected 4, got 3)

In [97]:
b['image'].shape

torch.Size([33, 3, 224, 224])